# Data 

### Data intake, formatting, and cleaning 

In [17]:
source("src/functions_and_things.r")

In [18]:
load_libraries()

## Pre polic data

June 1, 2024-December 31, 2024

* `subway_pre_df_jun` - June 1, 2024 - July 31, 2024
* `subway_pre_df_aug` - August 1, 2024 - September 31, 2024
* `subway_pre_df_oct` - October 1, 2024 - December 31, 2024

In [3]:
subway_pre_df_jun = read.csv("/Users/Local/statsproject/data/ignore/MTA_Jun2024_Subway_Hourly.csv")

subway_pre_df_aug = read.csv("/Users/Local/statsproject/data/ignore/MTA_Aug2024_Subway_Hourly.csv")

subway_pre_df_oct = read.csv("/Users/Local/statsproject/data/ignore/MTA_Oct2024_Subway_Hourly.csv")
subway_pre_df_oct |> select(-transfers) -> subway_pre_df_oct

In [4]:
subway_pre_df = rbind(subway_pre_df_oct, 
                      subway_pre_df_aug, 
                      subway_pre_df_jun)

subway_pre_df$time = 0

head(subway_pre_df)

,transit_timestamp,transit_mode,station_complex_id,station_complex,borough,payment_method,fare_class_category,ridership,latitude,longitude,time
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
1,01/01/2025 12:00:00 AM,subway,122,Graham Av (L),Brooklyn,metrocard,Metrocard - Full Fare,8,40.71457,-73.94405,0
2,01/01/2025 12:00:00 AM,subway,169,"Canal St (A,C,E)",Manhattan,omny,OMNY - Full Fare,171,40.72082,-74.00523,0
3,01/01/2025 12:00:00 AM,subway,173,"High St (A,C)",Brooklyn,metrocard,Metrocard - Full Fare,59,40.69934,-73.99053,0
4,01/01/2025 12:00:00 AM,subway,292,Fulton St (G),Brooklyn,omny,OMNY - Other,5,40.68712,-73.97537,0
5,01/01/2025 12:00:00 AM,subway,377,3 Av-138 St (6),Bronx,metrocard,Metrocard - Fair Fare,2,40.81047,-73.92614,0
6,01/01/2025 12:00:00 AM,subway,384,Burnside Av (4),Bronx,omny,OMNY - Full Fare,12,40.85345,-73.90768,0


In [5]:
head(subway_pre_df)

,transit_timestamp,transit_mode,station_complex_id,station_complex,borough,payment_method,fare_class_category,ridership,latitude,longitude,time
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
1,01/01/2025 12:00:00 AM,subway,122,Graham Av (L),Brooklyn,metrocard,Metrocard - Full Fare,8,40.71457,-73.94405,0
2,01/01/2025 12:00:00 AM,subway,169,"Canal St (A,C,E)",Manhattan,omny,OMNY - Full Fare,171,40.72082,-74.00523,0
3,01/01/2025 12:00:00 AM,subway,173,"High St (A,C)",Brooklyn,metrocard,Metrocard - Full Fare,59,40.69934,-73.99053,0
4,01/01/2025 12:00:00 AM,subway,292,Fulton St (G),Brooklyn,omny,OMNY - Other,5,40.68712,-73.97537,0
5,01/01/2025 12:00:00 AM,subway,377,3 Av-138 St (6),Bronx,metrocard,Metrocard - Fair Fare,2,40.81047,-73.92614,0
6,01/01/2025 12:00:00 AM,subway,384,Burnside Av (4),Bronx,omny,OMNY - Full Fare,12,40.85345,-73.90768,0


## Post policy data

January 1, 2025 - June 1st, 2025

In [7]:
subway_post_df = read.csv("/Users/Local/statsproject/data/ignore/MTA_Jan2025_Subway_Hourly.csv")
subway_post_df$time = 1

head(subway_post_df)

,transit_timestamp,transit_mode,station_complex_id,station_complex,borough,payment_method,fare_class_category,ridership,latitude,longitude,time
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
1,10/01/2024 12:00:00 AM,subway,69,25 Av (D),Brooklyn,omny,OMNY - Other,1,40.59771,-73.98683,1
2,10/01/2024 12:00:00 AM,subway,422,"Burke Av (2,5)",Bronx,metrocard,Metrocard - Unlimited 7-Day,1,40.87136,-73.86716,1
3,10/01/2024 12:00:00 AM,subway,602,"14 St-Union Sq (L,N,Q,R,W,4,5,6)",Manhattan,metrocard,Metrocard - Fair Fare,25,40.73574,-73.98995,1
4,10/01/2024 12:00:00 AM,subway,397,"86 St (4,5,6)",Manhattan,metrocard,Metrocard - Other,8,40.77949,-73.95559,1
5,10/01/2024 12:00:00 AM,subway,65,79 St (D),Brooklyn,metrocard,Metrocard - Unlimited 7-Day,1,40.61350,-74.00061,1
6,10/01/2024 12:00:00 AM,subway,405,23 St (6),Manhattan,metrocard,Metrocard - Unlimited 7-Day,7,40.73986,-73.98660,1


## Stacking pre and post datasets 

June 2024 - June 2025

24 million rows

In [8]:
df <- rbind(subway_pre_df, subway_post_df)

nrow(df)

[1] 24456218

## CBD Boolean

By latitude/ longitude

<img src="/Users/Local/statsproject/pics/cbd_subway_coordbox.jpg" width="400" height="400">

In [12]:
lat_high = 40.771372
long_high= -74.00782 

lat_low = 40.696999
long_low = -73.966776 

In [13]:
df$coords_cbd <- with(df, 
    latitude >= lat_low & latitude <= lat_high & 
    longitude <= long_low & longitude >= long_high)

Was the subway encoding close to the box one? 

In [14]:
sum(df$coords_cbd == TRUE)

[1] 3660800

In [36]:
sum(is.na(df$ridership))

df <- drop_na(df, ridership)

[1] 138693

## Aggregate dataset

In [37]:
df$ridership = as.integer(df$ridership)

str(df)

'data.frame':	24317525 obs. of  12 variables:
 $ transit_timestamp  : chr  "01/01/2025 12:00:00 AM" "01/01/2025 12:00:00 AM" "01/01/2025 12:00:00 AM" "01/01/2025 12:00:00 AM" ...
 $ transit_mode       : chr  "subway" "subway" "subway" "subway" ...
 $ station_complex_id : chr  "122" "169" "173" "292" ...
 $ station_complex    : chr  "Graham Av (L)" "Canal St (A,C,E)" "High St (A,C)" "Fulton St (G)" ...
 $ borough            : chr  "Brooklyn" "Manhattan" "Brooklyn" "Brooklyn" ...
 $ payment_method     : chr  "metrocard" "omny" "metrocard" "omny" ...
 $ fare_class_category: chr  "Metrocard - Full Fare" "OMNY - Full Fare" "Metrocard - Full Fare" "OMNY - Other" ...
 $ ridership          : int  8 171 59 5 2 12 1 98 2 2 ...
 $ latitude           : num  40.7 40.7 40.7 40.7 40.8 ...
 $ longitude          : num  -73.9 -74 -74 -74 -73.9 ...
 $ time               : num  0 0 0 0 0 0 0 0 0 0 ...
 $ coords_cbd         : logi  FALSE TRUE TRUE FALSE FALSE FALSE ...


In [38]:
df |> 
    mutate(
        date = as.Date(mdy_hms(transit_timestamp))) |>
    group_by(
        date, borough, coords_cbd) |> 
    summarise(
        ridership = sum(ridership)) |>
    select(date, borough, coords_cbd, ridership) -> df_daily

`summarise()` has grouped output by 'date', 'borough'. You can override using
the `.groups` argument.


In [44]:
head(df_daily, 8)

date,borough,coords_cbd,ridership
<date>,<chr>,<lgl>,<int>
2024-06-01,Bronx,FALSE,140211
2024-06-01,Brooklyn,FALSE,501778
2024-06-01,Brooklyn,TRUE,20848
2024-06-01,Manhattan,FALSE,383193
2024-06-01,Manhattan,TRUE,575531
2024-06-01,Queens,FALSE,345374
2024-06-02,Bronx,FALSE,107377
2024-06-02,Brooklyn,FALSE,403802


## Checks

In [42]:
nrow(df_daily)

[1] 2196

In [43]:
n_distinct(df_daily$date)

[1] 366